# Multi-ring segmentation API (CC-Lyon)

`MultiRingSegmenter` turns the hits of one event into rings (`segment`) or only counts them (`count_rings`).
It is built from a training output directory (`.hydra/config.yaml` + checkpoint) and a few inference keys.

The paths below are shared CC-Lyon files (model, digitizer mapping, test event), readable by the `hyperk` group.
**Not on CC-Lyon:** change `C`, `MODEL` and the event file to your copies.
Run it in an environment with torch and spconv, e.g. the `ml_image.sif` container ([available containers](../docs/cclyon_available_containers.md)).

In [1]:
import h5py
import numpy as np
import torch
from watchmal.engine.multiring.segmentation import MultiRingSegmenter

torch.set_num_threads(1)
C = "/sps/hyperk/melbaz/Reconstruction/const"
MODEL = f"{C}/watchmal/V3.0"   # training output directory of the provided model
cfg = dict(
    stats_cache_path=f"{MODEL}/feature_stats_2500_ctx.npz",   # feature normalisation of the training set
    digitizer_mapping_path=f"{C}/truehit_to_digihit_mapping_from_h5_firsthit.npz",
    lut_crossovers_path=f"{C}/lut_crossovers_optimized.txt",
    ring_pe_threshold=240.0,    # a ring exists if it owns this much charge (p.e.)
    ring_mass_threshold=500.0,  # ... or if its soft charge sum reaches this
    temperature=1.0,
    tta_views="r0, r1, r2, r3, r0z, r1z, r2z, r3z",  # test-time augmentation
)
seg = MultiRingSegmenter(MODEL, cfg, device="cpu")

One event of the test set: the earliest digit of each PMT, with raw WCSim times (as in training).

In [2]:
f = h5py.File("/sps/hyperk/melbaz/Simulation/output/HyperK_HybridmPMT_WithOD_Realistic/test/batch_0000/"
              "wcsim_output_multihit_with_digi_hit_and_trigger.h5", "r")
ev = 3
i = f["events/digi_hit_index"][ev:ev + 2]
d = f["events/digi_hits"][i[0]:i[1]]
o = np.lexsort((d["time_ns"], d["tube_id"]))
d = d[o[np.r_[True, d["tube_id"][o][1:] != d["tube_id"][o][:-1]]]]
xyz = np.stack([d["x"], d["y"], d["z"]], 1)
q, t, tubes = d["charge_pe"], d["time_ns"], d["tube_id"]
len(tubes), q.sum()

(6435, 13879.277)

`segment` returns a list of `Ring` (ring_id, tube_id, charge_pe, time_ns); `count_rings` the same number without splitting the charge.

In [3]:
rings = seg.segment(xyz, q, t, tubes)
for r in rings:
    print(r.ring_id, len(r.tube_id), round(r.total_charge, 1))
seg.count_rings(xyz, q, t, tubes) == len(rings)

2 2939 6846.6


True